In [19]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

In [20]:
load_dotenv()

api_key = os.getenv("OPENWEATHER_API_KEY")

In [21]:
url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Seoul",
    "appid": api_key,
    "units": "metric"
}

response = requests.get(url, params=params)

In [22]:
weather = response.json()
weather



{'coord': {'lon': 126.9778, 'lat': 37.5683},
 'weather': [{'id': 801,
   'main': 'Clouds',
   'description': 'few clouds',
   'icon': '02n'}],
 'base': 'stations',
 'main': {'temp': 23.81,
  'feels_like': 24.15,
  'temp_min': 23.81,
  'temp_max': 23.81,
  'pressure': 1012,
  'humidity': 73,
  'sea_level': 1012,
  'grnd_level': 1002},
 'visibility': 10000,
 'wind': {'speed': 1.87, 'deg': 220, 'gust': 2.94},
 'clouds': {'all': 14},
 'dt': 1782999937,
 'sys': {'country': 'KR', 'sunrise': 1782936873, 'sunset': 1782989833},
 'timezone': 32400,
 'id': 1835848,
 'name': 'Seoul',
 'cod': 200}

In [31]:
create_cities_table = """
CREATE TABLE IF NOT EXISTS cities(
    city_id SERIAL PRIMARY KEY,
    openweather_id INTEGER UNIQUE,
    city_name VARCHAR(50) NOT NULL,
    country CHAR(2) NOT NULL,
    latitude DECIMAL(8,5),
    longitude DECIMAL(8,5)
);
"""



In [27]:
load_dotenv()
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

In [28]:
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)

In [32]:
with engine.begin() as conn:
    result = conn.execute(text(create_cities_table))


In [ ]:
# from sqlalchemy import text

# drop_table_sql = """
# DROP TABLE IF EXISTS observations;
# """

# with engine.begin() as conn:
#     conn.execute(text(drop_table_sql))

In [33]:
insert_city_sql = """
INSERT INTO cities (
    openweather_id,
    city_name,
    country,
    latitude,
    longitude
)
VALUES (
    :openweather_id,
    :city_name,
    :country,
    :latitude,
    :longitude
)
"""

In [38]:
city_param = {
    "openweather_id": weather["id"],
    "city_name": weather["name"],
    "country": weather["sys"]["country"],
    "latitude": weather["coord"]["lat"],
    "longitude": weather["coord"]["lon"]
}


In [39]:
with engine.begin() as conn:
    conn.execute(text(insert_city_sql), city_param)